# 🎯 YOLO26 얼굴 감지 파인튜닝

가지고 있는 HSEmotion JSON 라벨을 YOLO 형식으로 변환해 YOLO26을 파인튜닝합니다.

## 작업 흐름

```
HSEmotion JSON (5개 파일)            YOLO 데이터셋
├── train_anger.json          →     ├── images/train/*.jpg
├── train_happy.json                ├── images/val/*.jpg
├── train_sadness.json              ├── labels/train/*.txt
├── train_panic.json                ├── labels/val/*.txt
└── train_blank.json                └── face_data.yaml
                                              ↓
                                    YOLO26 파인튜닝
                                              ↓
                                    best_face_model.pt
```

## 1단계 결과
- YOLO 형식 라벨 파일 자동 생성
- 얼굴 1클래스 (감정 라벨은 별도 모델이 처리)

## 2단계 결과
- 7,000장 사진으로 파인튜닝된 얼굴 감지 모델
- 영상 추론 시 더 정확한 얼굴 감지

## 1️⃣ 환경 설정

In [ ]:
# 패키지 설치
!pip install ultralytics tqdm pillow -q

print("✅ 패키지 설치 완료")

In [ ]:
# ═══════════════════════════════════════════════════
# 📁 경로 설정
# ═══════════════════════════════════════════════════

import os

# 입력 데이터
JSON_DIR        = "/workspace/user5/relabeled"            # JSON 파일 5개 위치
IMAGE_BASE_DIR  = "/workspace/user5/img/img(classify)"    # 원본 이미지 폴더

# 출력 (YOLO 형식 데이터셋)
YOLO_DATASET_DIR = "/workspace/user5/yolo_face_dataset"

# 학습 결과 저장
MODEL_OUTPUT_DIR = "/workspace/user5/yolo_finetuned"

# 학습 설정
EPOCHS     = 50
BATCH_SIZE = 32
IMG_SIZE   = 640      # YOLO 표준 입력 크기
DEVICE     = 0        # GPU 0번

print(f"📁 JSON: {JSON_DIR}")
print(f"📁 이미지: {IMAGE_BASE_DIR}")
print(f"📁 YOLO 데이터셋: {YOLO_DATASET_DIR}")
print(f"📁 모델 저장: {MODEL_OUTPUT_DIR}")

## 2️⃣ HSEmotion JSON → YOLO 형식 변환

각 JSON 파일에서 `bbox`를 읽어와 YOLO 형식 `.txt` 라벨로 변환합니다.

**YOLO 형식:**
```
class_id  center_x  center_y  width  height
```
- 모든 값 0~1로 정규화
- 한 이미지에 여러 얼굴이면 여러 줄
- 우리는 **얼굴 1클래스만** 사용 (`class_id = 0`)

In [ ]:
import json
import shutil
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

# 5개 감정 JSON 파일 매핑
JSON_FILES = {
    "anger":   "train_anger.json",
    "blank":   "train_blank.json",
    "happy":   "train_happy.json",
    "panic":   "train_panic.json",
    "sadness": "train_sadness.json",
}


def convert_bbox_to_yolo(bbox, img_w, img_h):
    """
    HSEmotion bbox → YOLO 형식 변환
    {minX, minY, maxX, maxY}  →  center_x, center_y, w, h (0~1 정규화)
    """
    min_x, min_y = bbox["minX"], bbox["minY"]
    max_x, max_y = bbox["maxX"], bbox["maxY"]

    # 좌표 보정 (이미지 범위 벗어나지 않게)
    min_x = max(0, min(min_x, img_w))
    min_y = max(0, min(min_y, img_h))
    max_x = max(0, min(max_x, img_w))
    max_y = max(0, min(max_y, img_h))

    # 중심 좌표 + 크기
    cx = (min_x + max_x) / 2 / img_w
    cy = (min_y + max_y) / 2 / img_h
    w  = (max_x - min_x) / img_w
    h  = (max_y - min_y) / img_h

    return cx, cy, w, h

print("✅ 변환 함수 준비 완료")

In [ ]:
def find_image(filename, emotion):
    """이미지 파일 경로 탐색 (train/val 둘 다 확인)"""
    candidates = [
        os.path.join(IMAGE_BASE_DIR, "train", emotion, filename),
        os.path.join(IMAGE_BASE_DIR, "val",   emotion, filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def setup_yolo_folders():
    """YOLO 데이터셋 폴더 구조 생성"""
    for split in ["train", "val"]:
        os.makedirs(os.path.join(YOLO_DATASET_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(YOLO_DATASET_DIR, "labels", split), exist_ok=True)
    print(f"✅ YOLO 폴더 구조 생성: {YOLO_DATASET_DIR}")


def convert_json_to_yolo():
    """
    5개 JSON → YOLO 라벨 + 이미지 복사
    train/val 비율 8:2로 자동 분할
    """
    setup_yolo_folders()

    stats = {"success": 0, "skipped_no_face": 0, "skipped_no_image": 0,
             "skipped_invalid": 0}

    for emotion, json_filename in JSON_FILES.items():
        json_path = os.path.join(JSON_DIR, json_filename)

        if not os.path.exists(json_path):
            print(f"⚠️  파일 없음: {json_path}")
            continue

        print(f"\n[{emotion}] {json_filename} 처리 중...")

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        images = data.get("images", [])

        for idx, item in enumerate(tqdm(images, desc=f"  {emotion}", leave=False)):
            filename = item.get("filename")

            # 1. 얼굴 감지 안 됐으면 스킵
            annot = item.get("annot_HSEmotion", {})
            if not annot.get("face_detected", False):
                stats["skipped_no_face"] += 1
                continue

            # 2. 이미지 파일 찾기
            img_path = find_image(filename, emotion)
            if img_path is None:
                stats["skipped_no_image"] += 1
                continue

            # 3. 이미지 크기 확인
            try:
                with Image.open(img_path) as im:
                    img_w, img_h = im.size
            except Exception:
                stats["skipped_invalid"] += 1
                continue

            # 4. bbox YOLO 변환
            bbox = item.get("bbox", {})
            if not bbox:
                stats["skipped_invalid"] += 1
                continue

            try:
                cx, cy, w, h = convert_bbox_to_yolo(bbox, img_w, img_h)
            except Exception:
                stats["skipped_invalid"] += 1
                continue

            # 유효성 체크
            if w <= 0 or h <= 0 or w > 1 or h > 1:
                stats["skipped_invalid"] += 1
                continue

            # 5. train/val 분할 (8:2)
            split = "val" if idx % 5 == 0 else "train"

            # 6. 이미지 복사 (확장자 통일을 위해 새 이름)
            base_name = Path(filename).stem
            new_img_name = f"{emotion}_{base_name}.jpg"
            new_lbl_name = f"{emotion}_{base_name}.txt"

            dest_img = os.path.join(YOLO_DATASET_DIR, "images", split, new_img_name)
            dest_lbl = os.path.join(YOLO_DATASET_DIR, "labels", split, new_lbl_name)

            # 이미지 복사
            shutil.copy2(img_path, dest_img)

            # YOLO 라벨 작성 (class_id=0: face)
            with open(dest_lbl, "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

            stats["success"] += 1

    print(f"""
╔═══════════════════════════════════════╗
║       JSON → YOLO 변환 완료            ║
╠═══════════════════════════════════════╣
║  성공:           {stats['success']:>5}장                ║
║  얼굴 미감지:    {stats['skipped_no_face']:>5}장 (스킵)         ║
║  이미지 없음:    {stats['skipped_no_image']:>5}장 (스킵)         ║
║  bbox 오류:      {stats['skipped_invalid']:>5}장 (스킵)         ║
╚═══════════════════════════════════════╝""")

    return stats

# 실행
stats = convert_json_to_yolo()

In [ ]:
# 변환 결과 검증
def verify_yolo_dataset():
    """YOLO 데이터셋이 잘 만들어졌는지 확인"""
    for split in ["train", "val"]:
        img_dir = os.path.join(YOLO_DATASET_DIR, "images", split)
        lbl_dir = os.path.join(YOLO_DATASET_DIR, "labels", split)

        imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")]
        lbls = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]

        print(f"  {split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")

    # 샘플 라벨 확인
    sample_lbl_dir = os.path.join(YOLO_DATASET_DIR, "labels", "train")
    sample_files = sorted(os.listdir(sample_lbl_dir))[:3]
    print(f"\n샘플 라벨:")
    for fname in sample_files:
        with open(os.path.join(sample_lbl_dir, fname)) as f:
            content = f.read().strip()
        print(f"  {fname}: {content}")

verify_yolo_dataset()

## 3️⃣ YOLO 학습 설정 파일 생성

`face_data.yaml` — YOLO에게 데이터 위치를 알려주는 설정 파일

In [ ]:
yaml_content = f"""# YOLO Face Detection Dataset
path: {YOLO_DATASET_DIR}
train: images/train
val: images/val

# 클래스 정의
nc: 1
names: ['face']
"""

yaml_path = os.path.join(YOLO_DATASET_DIR, "face_data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"✅ YAML 파일 생성: {yaml_path}\n")
print(yaml_content)

## 4️⃣ YOLO26 파인튜닝 학습

`yolo26n.pt`(nano)를 베이스로 얼굴 감지 모델 학습.

⏱️ 예상 시간: A40 기준 30분~1시간 (50 epochs)

In [ ]:
from ultralytics import YOLO

# YOLO26n 로드 (nano 버전, 가장 빠름)
print("YOLO26 베이스 모델 로드 중...")
model = YOLO("yolo26n.pt")
print(f"✅ 모델 로드 완료")
print(f"  파라미터: ~{sum(p.numel() for p in model.model.parameters())/1e6:.1f}M")

In [ ]:
# ═══════════════════════════════════════════════════
# 학습 시작
# ═══════════════════════════════════════════════════

results = model.train(
    data       = yaml_path,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    device     = DEVICE,
    project    = MODEL_OUTPUT_DIR,
    name       = "face_yolo26n",
    exist_ok   = True,

    # 데이터 증강 (극장 환경 대응)
    hsv_h      = 0.015,    # 색조 변화
    hsv_s      = 0.7,      # 채도
    hsv_v      = 0.4,      # 명도 (저조도 대응)
    degrees    = 10,       # 회전
    translate  = 0.1,      # 이동
    scale      = 0.5,      # 크기
    fliplr     = 0.5,      # 좌우반전
    mosaic     = 1.0,      # 모자이크 증강

    # 최적화
    optimizer  = "AdamW",
    lr0        = 0.001,
    weight_decay = 0.0005,
    patience   = 20,       # Early stopping
    save_period= 10,       # 10 epoch마다 저장

    # 로깅
    plots      = True,     # 학습 곡선 자동 저장
    verbose    = True,
)

print("\n✅ 학습 완료!")

In [ ]:
# 학습 결과 요약
import pandas as pd

# 결과 폴더 위치
result_dir = os.path.join(MODEL_OUTPUT_DIR, "face_yolo26n")
print(f"📁 결과 위치: {result_dir}")

# 최종 모델 경로
best_model_path = os.path.join(result_dir, "weights", "best.pt")
print(f"🏆 Best 모델: {best_model_path}")

# 학습 메트릭 (results.csv)
results_csv = os.path.join(result_dir, "results.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\n📊 최종 epoch 메트릭:")
    last = df.iloc[-1]
    for col in ["metrics/mAP50(B)", "metrics/mAP50-95(B)",
                "metrics/precision(B)", "metrics/recall(B)"]:
        if col in df.columns:
            print(f"  {col}: {last[col]:.4f}")

## 5️⃣ 학습된 모델 테스트

테스트 이미지로 얼굴 감지가 잘 되는지 확인

In [ ]:
import cv2
import matplotlib.pyplot as plt

# 학습된 best 모델 로드
best_model = YOLO(best_model_path)

# 테스트 이미지 (val 폴더에서 자동 선택)
val_img_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")
test_imgs = sorted(os.listdir(val_img_dir))[:6]

# 6장 테스트 + 시각화
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, fname in enumerate(test_imgs):
    img_path = os.path.join(val_img_dir, fname)
    results = best_model(img_path, verbose=False)

    # 시각화
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = float(box.conf[0])
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(img, f"{conf:.2f}", (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    axes[idx].imshow(img)
    axes[idx].set_title(fname[:25], fontsize=8)
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(result_dir, "test_results.png"), dpi=120)
plt.show()

print(f"\n✅ 테스트 결과 저장: {result_dir}/test_results.png")

## 6️⃣ 메인 시스템에 적용

`emotion_engagement_system.ipynb`에서 모델 경로만 바꾸면 돼요:

```python
# 기존 (사전학습된 일반 모델)
self.yolo = YOLO("yolov8n-face.pt")

# 변경 (내가 직접 파인튜닝한 모델)
self.yolo = YOLO("/workspace/user5/yolo_finetuned/face_yolo26n/weights/best.pt")
```

이러면 7,000장 데이터로 학습한 **우리 도메인 특화 모델**이 됩니다!

In [ ]:
# 최종 모델 경로 출력 (복사해서 메인 시스템에 붙여넣기)
print("="*55)
print("  파인튜닝 완료 — 최종 모델 경로")
print("="*55)
print(f"\n{best_model_path}\n")
print("메인 시스템 노트북의 EmotionVideoDetector에서:")
print(f'  self.yolo = YOLO("{best_model_path}")')